# Time-series analysis

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql-08-time-series.ipynb)

Time-series SQL is mostly three techniques: bucket timestamps into periods, fill the periods that have no data,
and compare each row with its neighbours in time. Most of the difficulty is in the edges: time zones, boundaries
and missing days.

Previous: [deduplication](sql-07-deduplication.ipynb). Next: [data modeling and optimization](sql-09-modeling-and-optimization.ipynb).

## Setup

The first code cells install PySpark if it is missing, detect Java, and start a local Spark session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=3.5,<4", "setuptools": "setuptools>=70"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
# Detect PySpark and Java. Spark needs a Java runtime that can actually start a JVM.
import importlib.util
import shutil
import subprocess

HAS_PYSPARK = importlib.util.find_spec("pyspark") is not None
HAS_JAVA = shutil.which("java") is not None

# `shutil.which` only proves the binary exists; probe that it can start a JVM.
JAVA_RUNS = False
if HAS_JAVA:
    try:
        probe = subprocess.run(["java", "-version"], capture_output=True, text=True, timeout=60)
        JAVA_RUNS = probe.returncode == 0
    except Exception:
        JAVA_RUNS = False

SPARK_AVAILABLE = HAS_PYSPARK and HAS_JAVA and JAVA_RUNS

print(f"PySpark installed : {HAS_PYSPARK}")
print(f"Java on PATH      : {HAS_JAVA}")
print(f"Java runnable     : {JAVA_RUNS}")
print(f"Spark available   : {SPARK_AVAILABLE}")
if not SPARK_AVAILABLE:
    print("SKIPPED: Spark is not available, so every example prints SKIPPED instead of running.")

In [ ]:
import os
import sys
import time
from datetime import date, datetime
from decimal import Decimal

# Spark timestamps and Python datetimes must agree on the clock, so pin the process to UTC.
os.environ["TZ"] = "UTC"
if hasattr(time, "tzset"):
    time.tzset()

if SPARK_AVAILABLE:
    from pyspark.sql import SparkSession

    os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
    spark = (
        SparkSession.builder.master("local[2]")
        .appName("sql-learning-path")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark session ready:", spark.version)
else:
    spark = None


def create_view(name, ddl, rows):
    """Register rows as a temporary view. Skipped when Spark is unavailable."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED view {name}: needs a Spark session.")
        return
    spark.createDataFrame(rows, ddl).createOrReplaceTempView(name)
    print(f"view {name}: {spark.table(name).count()} rows")


def _plain(value):
    """Spark returns DECIMAL for literal arithmetic; compare those as floats."""
    return float(value) if isinstance(value, Decimal) else value


def check(query, expected, label):
    """Show a query's result, then assert it equals the expected rows. Skipped without Spark."""
    if not SPARK_AVAILABLE:
        print(f"SKIPPED {label}: needs a Spark session.")
        return
    frame = spark.sql(query)
    frame.show(truncate=False)
    rows = [tuple(_plain(value) for value in row) for row in frame.collect()]
    assert rows == expected, f"{label}: got {rows}, expected {expected}"
    print(f"PASS {label}")

## Time zones and boundaries

This notebook pins the session to UTC. Keep timestamps in UTC inside the warehouse and convert only for display.
For ranges, use a half-open interval `[start, end)`: a row at exactly `end` belongs to the next period, so no row
is counted twice and none is lost.

## The example data

`orders` (dates) and `events` (timestamps), two small tables that cover the patterns below.

In [ ]:
create_view(
    "customers",
    "customer_id INT, name STRING, region STRING, signup_date DATE",
    [
        (1, "Acme Corp", "North", date(2022, 1, 5)),
        (2, "Beta LLC", "North", date(2022, 2, 10)),
        (3, "Cyan Inc", "South", date(2022, 3, 15)),
        (4, "Delta Co", "South", date(2022, 4, 20)),
        (5, "Echo Ltd", "West", date(2022, 5, 25)),
        (6, "Foxtrot GmbH", "West", date(2022, 6, 30)),
        (7, "Gamma SA", "East", date(2023, 1, 10)),
        (8, "Hotel BV", "East", date(2023, 2, 15)),
    ],
)

create_view(
    "orders",
    "order_id INT, customer_id INT, status STRING, amount INT, order_date DATE",
    [
        (1, 1, "completed", 100, date(2023, 1, 5)),
        (2, 1, "completed", 200, date(2023, 2, 10)),
        (3, 1, "completed", 150, date(2023, 3, 15)),
        (4, 2, "completed", 300, date(2023, 1, 20)),
        (5, 2, "cancelled", 250, date(2023, 2, 5)),
        (6, 2, "completed", 400, date(2023, 3, 20)),
        (7, 3, "completed", 120, date(2023, 1, 25)),
        (8, 3, "completed", 180, date(2023, 2, 28)),
        (9, 4, "pending", 220, date(2023, 2, 14)),
        (10, 4, "completed", 320, date(2023, 3, 30)),
        (11, 5, "completed", 500, date(2023, 1, 2)),
        (12, 5, "completed", 450, date(2023, 3, 5)),
        (13, 6, "completed", 275, date(2023, 2, 18)),
        (14, 6, "cancelled", 325, date(2023, 3, 25)),
    ],
)

In [ ]:
create_view(
    "events",
    "event_id INT, user_id INT, event_ts TIMESTAMP, event_type STRING",
    [
    (1, 1, datetime(2023, 1, 1, 9, 0, 0), 'login'),
    (2, 1, datetime(2023, 1, 2, 9, 5, 0), 'login'),
    (3, 1, datetime(2023, 1, 3, 8, 55, 0), 'login'),
    (4, 1, datetime(2023, 1, 5, 9, 0, 0), 'login'),
    (5, 1, datetime(2023, 1, 6, 9, 10, 0), 'login'),
    (6, 2, datetime(2023, 1, 1, 10, 0, 0), 'login'),
    (7, 2, datetime(2023, 1, 2, 10, 0, 0), 'login'),
    (8, 2, datetime(2023, 1, 4, 10, 0, 0), 'login'),
    (9, 3, datetime(2023, 2, 1, 10, 0, 0), 'view'),
    (10, 3, datetime(2023, 2, 1, 10, 5, 0), 'click'),
    (11, 3, datetime(2023, 2, 1, 10, 25, 0), 'view'),
    (12, 3, datetime(2023, 2, 1, 11, 30, 0), 'purchase'),
    (13, 3, datetime(2023, 2, 1, 11, 45, 0), 'view'),
    (14, 4, datetime(2023, 2, 1, 10, 0, 0), 'login'),
    (15, 4, datetime(2023, 2, 1, 10, 20, 0), 'view'),
    ],
)

## Bucketing into periods

`trunc(date, 'MM')` returns the first day of the month. Group by that value to get one row per month. For
weekly buckets use `'WEEK'`, and for quarters use `'QUARTER'`.

In [ ]:
check(
    """
    SELECT trunc(order_date, 'MM') AS month_start, COUNT(*) AS orders
    FROM orders
    GROUP BY trunc(order_date, 'MM')
    ORDER BY month_start
    """,
    [(date(2023, 1, 1), 4), (date(2023, 2, 1), 5), (date(2023, 3, 1), 5)],
    "orders per month",
)

## Filling gaps with a calendar spine

A day with no logins is absent from a `GROUP BY`, so a chart built from it shows a hole instead of a zero. Build
the calendar first with `sequence` and `explode`, then `LEFT JOIN` the activity onto it and use `COALESCE`.
Here 2023-01-07 has no logins at all.

In [ ]:
check(
    """
    WITH spine AS (
        SELECT explode(sequence(DATE '2023-01-01', DATE '2023-01-07', INTERVAL 1 DAY)) AS day
    ),
    daily AS (
        SELECT CAST(event_ts AS DATE) AS day, COUNT(*) AS logins
        FROM events
        WHERE event_type = 'login'
        GROUP BY CAST(event_ts AS DATE)
    )
    SELECT s.day, COALESCE(d.logins, 0) AS logins
    FROM spine s
    LEFT JOIN daily d ON d.day = s.day
    ORDER BY s.day
    """,
    [
        (date(2023, 1, 1), 2), (date(2023, 1, 2), 2), (date(2023, 1, 3), 1),
        (date(2023, 1, 4), 1), (date(2023, 1, 5), 1), (date(2023, 1, 6), 1),
        (date(2023, 1, 7), 0),
    ],
    "daily logins with zero-filled days",
)

## Gaps and islands: consecutive-day streaks

Within a run of consecutive days, subtracting the row number from the date gives a constant. That constant
identifies the run, so grouping by it finds each island. Deduplicate to one row per day first, or several
events on one day shift the row numbers.

`date_sub(d, n)` subtracts `n` days. Its second argument must be an integer, so the row number is cast.

In [ ]:
check(
    """
    WITH daily AS (
        SELECT DISTINCT user_id, CAST(event_ts AS DATE) AS d FROM events
    ),
    grouped AS (
        SELECT user_id, d,
               date_sub(d, CAST(ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY d) AS INT)) AS island
        FROM daily
    )
    SELECT user_id, MIN(d) AS start_day, MAX(d) AS end_day, COUNT(*) AS streak_days
    FROM grouped
    GROUP BY user_id, island
    ORDER BY user_id, start_day
    """,
    [
        (1, date(2023, 1, 1), date(2023, 1, 3), 3),
        (1, date(2023, 1, 5), date(2023, 1, 6), 2),
        (2, date(2023, 1, 1), date(2023, 1, 2), 2),
        (2, date(2023, 1, 4), date(2023, 1, 4), 1),
        (3, date(2023, 2, 1), date(2023, 2, 1), 1),
        (4, date(2023, 2, 1), date(2023, 2, 1), 1),
    ],
    "every streak (island)",
)

## Sessionization with `LAG`

A new session starts when the gap to the same user's previous event exceeds 30 minutes. Compute the gap with
`LAG`, flag the boundaries, and take a running sum of the flags to number the sessions. `unix_timestamp` converts
to seconds, so the gap in minutes is the difference divided by 60.

In [ ]:
check(
    """
    WITH ordered AS (
        SELECT user_id, event_ts, event_id,
               LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS prev_ts
        FROM events
    ),
    flagged AS (
        SELECT user_id, event_ts, event_id,
               CASE WHEN prev_ts IS NULL
                      OR (unix_timestamp(event_ts) - unix_timestamp(prev_ts)) / 60.0 > 30
                    THEN 1 ELSE 0 END AS starts_session
        FROM ordered
    )
    SELECT user_id, event_ts,
           SUM(starts_session) OVER (PARTITION BY user_id ORDER BY event_ts, event_id) AS session_no
    FROM flagged
    WHERE user_id IN (3, 4)
    ORDER BY user_id, event_ts
    """,
    [
        (3, datetime(2023, 2, 1, 10, 0), 1),
        (3, datetime(2023, 2, 1, 10, 5), 1),
        (3, datetime(2023, 2, 1, 10, 25), 1),
        (3, datetime(2023, 2, 1, 11, 30), 2),
        (3, datetime(2023, 2, 1, 11, 45), 2),
        (4, datetime(2023, 2, 1, 10, 0), 1),
        (4, datetime(2023, 2, 1, 10, 20), 1),
    ],
    "sessions for users 3 and 4",
)

## Cohorts and retention

A cohort is the group of customers who first appeared in the same period. For each cohort and each later month,
count the distinct customers who were active. Use `COUNT(DISTINCT ...)` so a customer with several orders in one
month is counted once.

In [ ]:
check(
    """
    WITH first_order AS (
        SELECT customer_id, MIN(date_format(order_date, 'yyyy-MM')) AS cohort_month
        FROM orders
        GROUP BY customer_id
    ),
    activity AS (
        SELECT DISTINCT customer_id, date_format(order_date, 'yyyy-MM') AS activity_month
        FROM orders
    )
    SELECT f.cohort_month, a.activity_month, COUNT(DISTINCT a.customer_id) AS active_customers
    FROM first_order f
    JOIN activity a ON a.customer_id = f.customer_id
    GROUP BY f.cohort_month, a.activity_month
    ORDER BY f.cohort_month, a.activity_month
    """,
    [
        ("2023-01", "2023-01", 4), ("2023-01", "2023-02", 3), ("2023-01", "2023-03", 3),
        ("2023-02", "2023-02", 2), ("2023-02", "2023-03", 2),
    ],
    "retention triangle",
)

## Cumulative distinct counts

"How many distinct customers had ordered by this date?" A correlated `COUNT(DISTINCT)` answers it, but it
recounts the history for every date. Spark does not allow `DISTINCT` inside a window, so compute each customer's
first order once and take a running sum of the new customers per day. That scales with customers, not with
dates times rows.

In [ ]:
check(
    """
    WITH first_order AS (
        SELECT customer_id, MIN(order_date) AS first_date
        FROM orders
        GROUP BY customer_id
    )
    SELECT first_date,
           COUNT(*) AS new_customers,
           SUM(COUNT(*)) OVER (ORDER BY first_date) AS cumulative_customers
    FROM first_order
    GROUP BY first_date
    ORDER BY first_date
    """,
    [
        (date(2023, 1, 2), 1, 1), (date(2023, 1, 5), 1, 2), (date(2023, 1, 20), 1, 3),
        (date(2023, 1, 25), 1, 4), (date(2023, 2, 14), 1, 5), (date(2023, 2, 18), 1, 6),
    ],
    "cumulative distinct customers",
)

## Takeaways

- Bucket with `trunc`, `date_format` or `date_trunc`, and say which time zone the buckets use.
- Build a calendar spine before reporting, so empty periods show as zero, not as missing.
- Gaps and islands and sessionization are both "compare with the previous row, then cumulate". Learn the pattern once.
- Use half-open intervals for every time range.